I'm looking to create a movie database graph. It needs to track who directed what, the cast members for each film, and how people have rated them

In [ ]:
# Import necessary libraries
import os
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm # For OpenAI support
from google.adk.sessions import InMemorySessionService
from google.adk.tools.tool_context import ToolContext
from google.adk.runners import Runner
from google.genai import types # For creating message Content/Parts
from typing import Optional, Dict, Any
import yaml

from helper import make_agent_caller, get_neo4j_import_dir

import warnings
warnings.filterwarnings("ignore")

import logging
logging.basicConfig(level=logging.CRITICAL)

from neo4j_for_adk import graphdb, tool_success, tool_error
from tools import (
    set_perceived_user_goal, 
    approve_perceived_user_goal, 
    get_approved_user_goal,
    list_available_files, 
    sample_file, 
    set_suggested_files, 
    get_suggested_files,
    approve_suggested_files)

# import tools defined in previous lesson
llm='gemini-3-flash-preview'

print("Libraries imported.")

/Users/dw/projects/Learn/Knowledge Graph/Build Knowledge Graph with Agentic-AI/.venv/lib/python3.12/site-packages/google/cloud/aiplatform/models.py:52: FutureWarning: Support for google-cloud-storage < 3.0.0 will be removed in a future version of google-cloud-aiplatform. Please upgrade to google-cloud-storage >= 3.0.0.
  from google.cloud.aiplatform.utils import gcs_utils


Libraries imported.


In [3]:
def load_agent_configs(filepath):
    with open(filepath, 'r') as f:
        data = yaml.safe_load(f)
    
    # Create a dictionary for easy access by agent name
    agent_map = {agent['name']: agent for agent in data['agents']}
    return agent_map

# Load the data
agent_configs = load_agent_configs('agents.yml')

# --- 1. Accessing the User Intent Agent ---
intent_cfg = agent_configs['user_intent_agent']
user_intent_instruction = f"{intent_cfg['role']}\n{intent_cfg['instructions']}"

print("--- USER INTENT AGENT INSTRUCTION ---")
print(user_intent_instruction)

# --- 2. Accessing the NER Agent ---
ner_cfg = agent_configs['ner_agent']
ner_instruction = f"{ner_cfg['role']}\n{ner_cfg['instructions']}"

print("--- NER AGENT INSTRUCTION ---")
print(ner_instruction)

# --- 3. Accessing the File Suggestion Agent ---
ner_cfg = agent_configs['file_suggestion_agent']
file_suggestion_instruction = f"{ner_cfg['role']}\n{ner_cfg['instructions']}"

print("--- FILE SUGGESTION AGENT INSTRUCTION ---")
print(file_suggestion_instruction)

--- USER INTENT AGENT INSTRUCTION ---
Knowledge Graph Use Case Architect
# Persona
You are an expert at knowledge graph use cases. Your tone is consultative and insightful.

# Conversational Hints
If the user is unsure, suggest classic use cases:
- Social networks (relationships)
- Logistics networks (suppliers/partners)
- Recommendation systems (products/customers)
- Fraud detection (transaction patterns)
- Pop-culture (movies/music/books)

# Output Definition
A user goal must have:
- kind_of_graph: 1-3 words (e.g., "Social Network")
- description: A few sentences explaining the intent (e.g., "Analyzing professional relationships.")

# Chain of Thought
1. Understand the user's goal through dialogue.
2. Ask clarifying questions as needed.
3. Use 'set_perceived_user_goal' to record your current understanding.
4. Present this to the user for confirmation.
5. Once agreed, use 'approve_perceived_user_goal' to finalize state.

--- NER AGENT INSTRUCTION ---
Named Entity Strategist
# Persona


In [4]:
# add the tools to a list
user_intent_agent_tools = [set_perceived_user_goal, approve_perceived_user_goal]

In [6]:
# Finally, construct the agent
user_intent_agent = Agent(
    name="user_intent_agent_v1", # a unique, versioned name
    model=llm, # defined earlier in a variable
    description="Helps the user ideate on a knowledge graph use case.", # used for delegation
    instruction=user_intent_instruction, # the complete instructions you composed earlier
    tools=user_intent_agent_tools, # the list of tools
)

print(f"Agent '{user_intent_agent.name}' created.")
print(f"Instruction '{user_intent_instruction}")

Agent 'user_intent_agent_v1' created.
Instruction 'Knowledge Graph Use Case Architect
# Persona
You are an expert at knowledge graph use cases. Your tone is consultative and insightful.

# Conversational Hints
If the user is unsure, suggest classic use cases:
- Social networks (relationships)
- Logistics networks (suppliers/partners)
- Recommendation systems (products/customers)
- Fraud detection (transaction patterns)
- Pop-culture (movies/music/books)

# Output Definition
A user goal must have:
- kind_of_graph: 1-3 words (e.g., "Social Network")
- description: A few sentences explaining the intent (e.g., "Analyzing professional relationships.")

# Chain of Thought
1. Understand the user's goal through dialogue.
2. Ask clarifying questions as needed.
3. Use 'set_perceived_user_goal' to record your current understanding.
4. Present this to the user for confirmation.
5. Once agreed, use 'approve_perceived_user_goal' to finalize state.



In [7]:
# NOTE: if re-running the session, come back here to re-initialize the agent
user_intent_caller = await make_agent_caller(user_intent_agent)

In [11]:
# Run the Initial Conversation

session_start = await user_intent_caller.get_session()
print(f"Session Start: {session_start.state}") # expect this to be empty

# We need an async function to await for each conversation
async def run_conversation():
    # start things off by describing your goal
    await user_intent_caller.call("""I'd like a bill of materials graph (BOM graph) which includes all levels from suppliers to finished product, 
    which can support root-cause analysis.""") 

    if "perceived_user_goal" not in session_start.state:
        # the LLM may have asked a clarifying question. offer some more details
        await user_intent_caller.call("""I'm concerned about possible manufacturing or supplier issues.""")        

    # Optimistically presume approval.
    await user_intent_caller.call("Approve that goal.", True)

await run_conversation()

session_end = await user_intent_caller.get_session()

Session Start: {}

>>> User Query: I'd like a bill of materials graph (BOM graph) which includes all levels from suppliers to finished product, 
    which can support root-cause analysis.
<<< Agent Response: That sounds like a powerful application for a knowledge graph. Mapping a Bill of Materials (BOM) this way allows you to traverse complex dependencies that traditional relational databases often struggle with—especially when you need to jump across dozens of levels instantly.

To make this effective for **root-cause analysis**, I’ve captured your goal as follows:

*   **Kind of Graph:** Bill of Materials (BOM)
*   **Description:** A multi-level supply chain and manufacturing graph mapping the hierarchy from raw materials and sub-tier suppliers up to the finished product. This graph is intended to facilitate root-cause analysis by tracing quality issues or delays back through the assembly chain to specific components or vendors.

Does this accurately reflect what you are looking to b

In [13]:
# List of tools for the file suggestion agent
file_suggestion_agent_tools = [
    get_approved_user_goal, 
    list_available_files, 
    sample_file, 
    set_suggested_files, 
    get_suggested_files,
    approve_suggested_files
]

In [14]:
# Finally, construct the agent

file_suggestion_agent = Agent(
    name="file_suggestion_agent_v1",
    model=llm, # defined earlier in a variable
    description="Helps the user select files to import.",
    instruction=file_suggestion_instruction,
    tools=file_suggestion_agent_tools,
)

print(f"Agent '{file_suggestion_agent.name}' created.")
print(f"Instruction '{file_suggestion_instruction}")

Agent 'file_suggestion_agent_v1' created.
Instruction 'Data Relevance Critic
# Persona
You are a constructive critic reviewing a list of files for graph construction.

# Task Constraints
- Only consider structured data files like CSV, JSON, or Excel.
- Use 'get_approved_user_goal' to align with the project scope.

# Chain of Thought
1. Fetch the approved goal and list available files using 'list_available_files'.
2. If a file's content is unclear, use 'sample_file' to inspect headers/rows.
3. Evaluate relevance and record suggestions via 'set_suggested_files'.
4. Ask the user to approve the list. 
5. Iterate based on feedback; if approved, use 'approve_suggested_files'.



In [15]:
from helper import make_agent_caller

file_suggestion_caller = await make_agent_caller(file_suggestion_agent)

In [16]:
# Run the Initial Conversation

# nudge the agent to look for files. in the full system, this will be the natural next step
await file_suggestion_caller.call("What files can we use for import?")

session_end = await file_suggestion_caller.get_session()

print("\n---\n")

# expect that the agent has listed available files
print("Available files: ", session_end.state["all_available_files"])

# the suggested files should be reasonable looking CSV files
print("Suggested files: ", session_end.state["suggested_files"])


>>> User Query: What files can we use for import?
<<< Agent Response: I am currently unable to retrieve the list of files from the import directory due to a technical issue. Additionally, I don't see an approved project goal yet.

To help you select the right files for your graph construction, could you please:
1.  **Define your goal:** What are you trying to achieve with this graph? (e.g., "Analyze customer purchase patterns" or "Map social network interactions").
2.  **List the files you have available:** If you know the filenames in your import directory, please share them so I can help evaluate their relevance once the connection is restored.

Once I have a goal and can access the file list, I will sample the contents and suggest the most relevant files for your project.

---



KeyError: 'all_available_files'

In [ ]:
# Agree with the file suggestions
await file_suggestion_caller.call("Yes, let's do it")

session_end = await file_suggestion_caller.get_session()

print("\n---\n")

print("Approved files: ", session_end.state["approved_files"])